# Pertemuan 4 — Mendiagnosis Kualitas Data
### Workbook belajar mandiri

**Exploratory Data Analysis (KMTI21133)**  
**Sub-CPMK4:** Mahasiswa mampu mendiagnosis masalah kualitas data: *missing values*, duplikasi, inkonsistensi, dan kesalahan tipe data (C4).

---

> **Dosen berhalangan hadir pada pertemuan 4 dan 5.** Kelas didampingi asisten, dan workbook ini dirancang agar
> dapat Anda selesaikan sendiri — di kelas bersama asisten, atau sepenuhnya mandiri di luar kelas.

### Cara memakai workbook ini

| | |
|---|---|
| **Perkiraan waktu** | 150 menit. Bagian 1–5 sekitar 75 menit, Bagian 6–9 sekitar 50 menit, Bagian 10 sekitar 25 menit. |
| **Sel penjelasan** | Dibaca. Tidak ada yang perlu dijalankan. |
| **Sel CONTOH** | Dijalankan apa adanya, lalu amati keluarannya. |
| **Sel TUGAS** | Anda isi sendiri, lalu jalankan sel `cek(...)` di bawahnya. |
| **Umpan balik** | Sel `cek(...)` akan menjawab **BENAR** atau **BELUM** beserta petunjuk. Jawaban tidak ditampilkan, jadi silakan mencoba sebanyak yang Anda perlu. |
| **Kalau mentok** | Lihat Bagian *Kalau Anda mentok* di akhir workbook sebelum bertanya. |

### Yang Anda hasilkan di akhir sesi

1. **Profil kualitas data** untuk dataset latihan (Bagian 9).
2. **Profil kualitas data awal untuk dataset proyek tim** (Bagian 10) — ini bagian pertama dari **M2** yang dikumpulkan pada pertemuan 5.

> **Jangan memperbaiki apa pun hari ini.** Pertemuan 4 hanya mendiagnosis. Perbaikan dikerjakan pada pertemuan 5.
> Memperbaiki sebelum mendiagnosis adalah cara tercepat menghilangkan bukti.

---
## Bagian 0 — Persiapan

Dataset: **`transaksi_koperasi_2025.csv`** — catatan transaksi koperasi mahasiswa, Agustus–November 2025.

> Seluruh isinya **sintetis**. Tidak ada orang nyata di dalamnya. Namun cacat datanya dirancang menyerupai
> data operasional sungguhan, termasuk jenis cacat yang paling sering luput dari perhatian.

In [ ]:
import pandas as pd
import numpy as np
import hashlib, os

pd.set_option('display.max_columns', 30)
pd.set_option('display.width', 140)

FILE = 'transaksi_koperasi_2025.csv'
if not os.path.exists(FILE):
    try:
        from google.colab import files
        files.upload()
    except ImportError:
        raise FileNotFoundError('Letakkan CSV di folder yang sama dengan notebook ini.')

df = pd.read_csv(FILE)
print('Berhasil dimuat:', df.shape)

### Alat pemeriksa jawaban
Jalankan sel berikut satu kali. Setelah itu Anda bisa memanggil `cek('T1_baris', jawaban_saya)` kapan saja.

In [ ]:
_K = {
    'T1_baris': '50a0a04f2d',
    'T1_kolom': 'b17ef6d19c',
    'T2_kosong': '3a481e7283',
    'T2_kolom': 'ef2d127de3',
    'T3_rating': '0e17daca5f',
    'T4_jasa': '43b87f618c',
    'T5_dup': 'c2356069e9',
    'T5_dupid': '8241649609',
    'T6_kat': '4ec9599fc2',
    'T6_katbersih': 'ef2d127de3',
    'T7_rp': '1e472b39b1',
    'T8_jumlah': '35135aaa6c',
    'T8_diskon': '19581e27de',
    'T9_anggota': 'aae0212936',
    'T9_strip': '7c252ab334',
    'T10_bisnis': '44c8031cb0',
    'T12_bulan': '4b227777d4',
    'T12_rusak': '48ce32e8ec'
}

_P = {
    'T1_baris': 'Pakai df.shape[0] atau len(df).',
    'T1_kolom': 'Pakai df.shape[1].',
    'T2_kosong': 'df.isna().sum() memberi per kolom; jumlahkan lagi dengan .sum().',
    'T2_kolom': 'Hitung berapa kolom yang nilai isna().sum()-nya lebih dari 0.',
    'T3_rating': 'Bandingkan kolom rating_layanan dengan -999, lalu jumlahkan.',
    'T4_jasa': 'Saring baris berkategori Jasa (rapikan dulu spasi dan hurufnya), lalu .isna().mean()*100 pada berat_kg. Bulatkan 1 desimal.',
    'T5_dup': 'df.duplicated().sum() menghitung baris yang SELURUH kolomnya sama.',
    'T5_dupid': 'Pakai df[\'id_transaksi\'].duplicated(keep=False).sum() agar kedua pasangannya ikut terhitung.',
    'T6_kat': 'Hitung nunique() pada kolom kategori apa adanya, tanpa dirapikan.',
    'T6_katbersih': 'Rapikan: .str.strip().str.title(), lalu satukan \'Minumam\'->\'Minuman\' dan \'Alat Tulis\'->\'Atk\'.',
    'T7_rp': 'Ubah ke teks dengan .astype(str), lalu .str.startswith(\'Rp\').',
    'T8_jumlah': 'Hitung baris dengan jumlah kurang dari atau sama dengan nol.',
    'T8_diskon': 'Gabungkan dua kondisi dengan | dan bungkus masing-masing dengan tanda kurung.',
    'T9_anggota': 'nunique() pada id_anggota apa adanya.',
    'T9_strip': 'nunique() setelah .str.strip().',
    'T10_bisnis': 'Kerjakan hanya pada baris yang total_bayar-nya sudah berupa angka. Bandingkan dengan (jumlah x harga x (1 - diskon/100)).round().',
    'T12_bulan': 'Setelah konversi yang benar, pakai .dt.month.nunique(). Data ini hanya mencakup beberapa bulan saja.',
    'T12_rusak': 'Bandingkan hasil konversi yang benar dengan hasil pd.to_datetime(..., format=\'mixed\', dayfirst=True) memakai !=, lalu jumlahkan.'
}

def _h(x):
    return hashlib.sha256(str(x).strip().lower().encode()).hexdigest()[:10]

def cek(tugas, jawaban):
    """Periksa jawaban Anda tanpa menampilkan kuncinya."""
    if tugas not in _K:
        print('Nama tugas tidak dikenal:', tugas); return
    if _h(jawaban) == _K[tugas]:
        print(f'BENAR   {tugas} = {jawaban}')
    else:
        print(f'BELUM   {tugas} = {jawaban}')
        print('        Petunjuk:', _P.get(tugas, '-'))

def skor():
    """Tidak melacak apa pun; sekadar pengingat daftar tugas."""
    print('Daftar tugas:', ', '.join(sorted(_K)))

print('Alat pemeriksa siap. Contoh pemakaian: cek("T1_baris", 1000)')

---
## Bagian 1 — Lima dimensi kualitas data

Sebelum menyentuh kode, kenali apa saja yang sedang kita cari. Kualitas data biasanya dipecah menjadi lima dimensi.

| Dimensi | Pertanyaannya | Contoh masalah |
|---|---|---|
| **Kelengkapan** | Apakah ada nilai yang hilang? | Kolom kosong, atau placeholder seperti `-999` dan `N/A` |
| **Keunikan** | Apakah ada yang tercatat dua kali? | Baris duplikat persis maupun nyaris sama |
| **Konsistensi** | Apakah hal yang sama ditulis dengan cara yang sama? | `ATK`, `Atk`, ` ATK `, dan `Alat Tulis` sebagai empat kategori berbeda |
| **Validitas** | Apakah nilainya mungkin terjadi? | Jumlah barang negatif, diskon 150%, tanggal di masa depan |
| **Akurasi** | Apakah nilainya benar? | Total bayar tidak sama dengan jumlah × harga |

Urutan pemeriksaannya mengikuti urutan tabel ini, karena masalah di baris atas sering menyebabkan
salah tafsir pada baris bawah. Misalnya, menghitung rata-rata rating tanpa menyadari adanya `-999`
akan menghasilkan angka yang masuk akal secara tampilan tetapi salah total.

### Satu prinsip yang menentukan segalanya

> **Data yang kelihatan rapi belum tentu benar, dan pandas tidak akan memberi tahu Anda.**

`df.head()` akan tampil cantik untuk data seburuk apa pun. Satu-satunya cara mengetahui kondisi data
adalah memeriksanya satu dimensi demi satu dimensi, dengan sengaja.

---
## Bagian 2 — Potret awal

Tujuh perintah ini adalah hal pertama yang dijalankan pada dataset apa pun, selamanya.

In [ ]:
# CONTOH 2.1 — tampilan permukaan
df.head(5)

In [ ]:
# CONTOH 2.2 — ukuran dan tipe
print('Ukuran:', df.shape)
print()
df.info()

> **Amati kolom `Dtype` di atas.** Tiga hal yang ganjil:
> 
> - `tanggal_transaksi` bertipe `object`, padahal isinya tanggal.
> - `total_bayar` bertipe `object`, padahal isinya uang.
> - `kategori` dan `metode_bayar` bertipe `object` — wajar untuk teks, tetapi perlu dicek isinya.
> 
> Kolom uang yang terbaca sebagai teks hampir selalu berarti ada nilai berformat di dalamnya,
> misalnya `Rp 24.000`. Satu nilai seperti itu sudah cukup membuat seluruh kolom menjadi teks.

In [ ]:
# CONTOH 2.3 — ringkasan numerik
df.describe().round(2)

> **Lihat baris `min` pada `rating_layanan`.** Nilainya `-999`, dan `mean`-nya ikut menjadi negatif.
> Ini bukan kesalahan hitung pandas; pandas memang diminta merata-ratakan angka `-999`.
> Angka seperti ini disebut **placeholder**: tanda data kosong yang ditulis sebagai angka,
> sehingga lolos dari `isna()`.
> 
> Perhatikan juga `min` pada `jumlah` dan `max` pada `diskon_persen`.

In [ ]:
# CONTOH 2.4 — ringkasan kolom teks
df.describe(include='object').T

### TUGAS 1 — ukuran dataset

In [ ]:
# TUGAS 1
t1_baris = None   # ganti dengan jumlah baris
t1_kolom = None   # ganti dengan jumlah kolom

cek('T1_baris', t1_baris)
cek('T1_kolom', t1_kolom)

---
## Bagian 3 — Kelengkapan: nilai kosong dan penyamarannya

Ada dua macam nilai kosong, dan hanya satu yang terlihat oleh `isna()`.

| Jenis | Contoh | Terdeteksi `isna()`? |
|---|---|---|
| Kosong sejati | `NaN`, sel kosong di CSV | Ya |
| Placeholder angka | `-999`, `0` yang berarti tidak diisi, `9999` | **Tidak** |
| Placeholder teks | `N/A`, `-`, `null`, `tidak diketahui`, `?` | **Tidak** |

Placeholder berbahaya justru karena ia terlihat seperti data sungguhan.

In [ ]:
# CONTOH 3.1 — kosong sejati
kosong = df.isna().sum()
kosong[kosong > 0].to_frame('jumlah_kosong').assign(
    persen=lambda d: (d.jumlah_kosong / len(df) * 100).round(1))

In [ ]:
# CONTOH 3.2 — berburu placeholder teks
CURIGA = ['n/a', 'na', '-', '', 'null', 'none', 'tidak diketahui', 'unknown', '?', '#n/a']

for kol in df.select_dtypes(include='object').columns:
    nilai = df[kol].astype(str).str.strip().str.lower()
    ketemu = nilai[nilai.isin(CURIGA)].value_counts()
    if len(ketemu):
        print(f'{kol}:')
        print(ketemu.to_string(), '\n')

In [ ]:
# CONTOH 3.3 — berburu placeholder angka
# Caranya: lihat nilai ekstrem pada tiap kolom numerik
for kol in df.select_dtypes(include='number').columns:
    s = df[kol]
    print(f'{kol:18s} min={s.min():12,.1f}  max={s.max():14,.1f}  '
          f'nilai terkecil unik: {sorted(s.dropna().unique())[:3]}')

> **Pola yang perlu dihafal.** Placeholder angka hampir selalu berupa bilangan yang mencolok:
> `-1`, `-999`, `0`, `9999`, `99999`, atau tahun `1900`. Kalau `min` atau `max` sebuah kolom terasa aneh,
> periksa nilai uniknya sebelum menganggapnya data sungguhan.

### TUGAS 2 — kelengkapan

In [ ]:
# TUGAS 2
t2_kosong = None   # TOTAL sel kosong di seluruh dataframe
t2_kolom  = None   # BERAPA KOLOM yang punya setidaknya satu nilai kosong

cek('T2_kosong', t2_kosong)
cek('T2_kolom', t2_kolom)

### TUGAS 3 — placeholder angka

In [ ]:
# TUGAS 3
t3_rating = None   # berapa baris yang rating_layanan-nya bernilai -999

cek('T3_rating', t3_rating)

---
## Bagian 4 — Mengapa data itu kosong: MCAR, MAR, MNAR

Menemukan nilai kosong baru separuh pekerjaan. Separuh lainnya adalah menebak **mengapa** kosong,
karena alasannya menentukan apakah Anda boleh menghapus baris, boleh mengisi, atau tidak boleh keduanya.

| Istilah | Artinya | Contoh | Akibatnya |
|---|---|---|---|
| **MCAR** | Kosong betul-betul acak | Kertas struk sobek | Menghapus baris aman, hanya kehilangan ukuran sampel |
| **MAR** | Kosong bergantung pada kolom lain **yang terlihat** | Berat barang kosong untuk produk non-fisik | Bisa ditangani; isi per kelompok, atau akui bahwa kolom itu memang tidak berlaku |
| **MNAR** | Kosong bergantung pada **nilainya sendiri** yang tidak terlihat | Saldo besar sengaja tidak ditampilkan | Paling berbahaya; mengisi rata-rata akan menyesatkan |

> **Hal terpenting di bagian ini:** MAR dapat dibuktikan dari data. MCAR dan MNAR **tidak bisa dibedakan**
> hanya dengan melihat data, karena nilai yang hilang tidak dapat diperiksa. Membedakannya memerlukan
> pengetahuan tentang bagaimana data itu dikumpulkan.

In [ ]:
# CONTOH 4.1 — alat bantu: apakah kekosongan terkait kolom lain?
def pola_kosong(data, kolom_kosong, kolom_pembanding):
    """Persentase kekosongan kolom_kosong untuk tiap nilai kolom_pembanding."""
    return (data.groupby(kolom_pembanding)[kolom_kosong]
                .apply(lambda s: round(s.isna().mean() * 100, 1))
                .sort_values(ascending=False)
                .to_frame('persen_kosong'))

# rapikan kategori lebih dulu supaya pengelompokannya bermakna
kat_rapi = (df['kategori'].astype(str).str.strip().str.title()
              .replace({'Minumam': 'Minuman', 'Alat Tulis': 'Atk'}))
df_bantu = df.assign(kategori_rapi=kat_rapi)

pola_kosong(df_bantu, 'berat_kg', 'kategori_rapi')

> **Ini MAR, dan buktinya jelas.** `berat_kg` kosong 100% untuk Jasa dan Pulsa — keduanya memang tidak
> punya berat. Kekosongannya bergantung pada `kategori`, kolom yang bisa kita lihat.
> 
> Tindakan yang tepat **bukan** mengisi dengan rata-rata, melainkan mengakui bahwa kolom ini hanya
> berlaku untuk barang fisik. Mengisi berat Pulsa dengan 0,3 kg akan menciptakan data palsu.

In [ ]:
# CONTOH 4.2 — bandingkan dengan kolom lain
pola_kosong(df_bantu, 'catatan_kasir', 'kategori_rapi')

> **Ini tampak MCAR.** Persentasenya mirip di semua kategori. Tidak ada kolom teramati yang menjelaskan
> kekosongannya, jadi menghapus atau mengabaikan kolom ini relatif aman.
> 
> Catatan: sebagian kekosongan ini sebenarnya berasal dari sel yang memang dibiarkan kosong oleh kasir
> (teks kosong), yang oleh `read_csv` dibaca menjadi `NaN`. Teks kosong dan data hilang adalah dua hal
> berbeda, tetapi CSV tidak bisa membedakannya. Ini keterbatasan format, bukan kesalahan Anda.

In [ ]:
# CONTOH 4.3 — kasus yang menipu
for kol in ['kategori_rapi', 'kasir', 'metode_bayar']:
    print(f'--- saldo_simpanan kosong menurut {kol} ---')
    print(pola_kosong(df_bantu, 'saldo_simpanan', kol).head(5).to_string(), '\n')

In [ ]:
# CONTOH 4.4 — dan pembandingan rata-rata kolom numerik
m = df['saldo_simpanan'].isna()
banding = pd.DataFrame({
    'saat_kosong': df.loc[m, ['jumlah', 'harga_satuan', 'diskon_persen']].mean().round(2),
    'saat_terisi': df.loc[~m, ['jumlah', 'harga_satuan', 'diskon_persen']].mean().round(2),
})
print(banding.to_string())
print()
print('Saldo yang TERISI — median:', f"{df['saldo_simpanan'].median():,.0f}",
      '| maksimum:', f"{df['saldo_simpanan'].max():,.0f}")

> **Baca pelan-pelan, ini inti pertemuan hari ini.**
> 
> Setiap pemeriksaan di atas mengatakan kekosongan `saldo_simpanan` tidak terkait kolom mana pun.
> Secara statistik, kolom ini **tampak MCAR**, dan Anda akan tergoda mengisinya dengan median.
> 
> Sekarang perhatikan satu hal yang tidak muncul di pemeriksaan mana pun: saldo tertinggi yang
> *masih terlihat* hanya sekitar 3,5 juta. Untuk koperasi dengan ratusan anggota, tidak adanya satu pun
> saldo besar adalah hal yang ganjil. Dugaan yang masuk akal: anggota bersaldo besar memilih tidak
> menampilkan saldonya.
> 
> Kalau dugaan itu benar, kolom ini **MNAR**, dan mengisinya dengan median akan membuat rata-rata saldo
> koperasi tampak jauh lebih kecil daripada kenyataannya. Tidak ada satu pun uji statistik yang bisa
> memastikan hal ini. Yang bisa memastikannya hanya satu: **bertanya kepada pemilik data**.

### TUGAS 4 — membuktikan MAR

In [ ]:
# TUGAS 4
# Berapa persen baris berkategori Jasa yang berat_kg-nya kosong?
# Bulatkan satu desimal. Gunakan kategori yang sudah dirapikan.
t4_jasa = None

cek('T4_jasa', t4_jasa)

**Pertanyaan renungan (tulis jawabannya di sini, tidak ada sel cek):**

Untuk dataset proyek tim Anda, sebutkan satu kolom yang Anda duga MNAR, dan jelaskan
mengapa dugaan itu tidak bisa dibuktikan dari datanya sendiri.

_(tulis di sini)_

---
## Bagian 5 — Keunikan: duplikat persis dan duplikat nyaris

Duplikat persis mudah ditemukan. Yang merepotkan adalah duplikat nyaris: baris yang merujuk kejadian
yang sama tetapi berbeda sedikit, misalnya beda huruf besar-kecil atau satu kolom kosong.

In [ ]:
# CONTOH 5.1 — duplikat persis
print('Baris yang seluruh kolomnya sama:', df.duplicated().sum())
df[df.duplicated(keep=False)].sort_values('id_transaksi').head(6)

In [ ]:
# CONTOH 5.2 — duplikat pada kunci utama
# id_transaksi seharusnya unik. Kalau tidak, ada yang salah.
dup_id = df['id_transaksi'].duplicated(keep=False)
print('Baris dengan id_transaksi berulang:', dup_id.sum())
print('id_transaksi kosong             :', df['id_transaksi'].isna().sum())

contoh = df[dup_id].sort_values('id_transaksi').head(4)
contoh[['id_transaksi', 'kasir', 'catatan_kasir', 'total_bayar']]

> **Perhatikan contoh di atas.** Ada pasangan baris dengan `id_transaksi` yang sama, tetapi nama kasir
> ditulis berbeda kapitalisasinya dan `catatan_kasir`-nya kosong di salah satu baris. `duplicated()`
> biasa tidak menangkapnya, karena bagi pandas kedua baris itu memang berbeda.
> 
> Cara menemukannya: cari duplikat pada **kolom kunci saja**, bukan seluruh kolom.

In [ ]:
# CONTOH 5.3 — duplikat berdasarkan sebagian kolom
kunci = ['id_transaksi']
print('Duplikat menurut kunci :', df.duplicated(subset=kunci, keep=False).sum())
print('Duplikat seluruh kolom :', df.duplicated(keep=False).sum())
print()
print('Selisihnya adalah duplikat NYARIS, yang justru paling sering luput.')

### TUGAS 5 — keunikan

In [ ]:
# TUGAS 5
t5_dup   = None   # jumlah baris duplikat menurut df.duplicated() -- perhatikan perilaku default keep='first'
t5_dupid = None   # jumlah baris yang id_transaksi-nya berulang, dengan keep=False

cek('T5_dup', t5_dup)
cek('T5_dupid', t5_dupid)

---
## Bagian 6 — Konsistensi: satu hal, banyak tulisan

Masalah paling umum pada data yang diketik manusia, dan paling sering tidak disadari karena
dampaknya baru muncul saat agregasi.

In [ ]:
# CONTOH 6.1 — berapa kategori sebenarnya?
df['kategori'].value_counts(dropna=False)

> Koperasi ini hanya punya lima kategori. Daftar di atas jauh lebih panjang.
> Empat penyebabnya: **huruf besar-kecil**, **spasi di awal atau akhir**, **sinonim**, dan **salah ketik**.

In [ ]:
# CONTOH 6.2 — memisahkan keempat penyebab
s = df['kategori'].astype(str)
print('Nilai unik apa adanya            :', s.nunique())
print('Setelah dibuang spasinya         :', s.str.strip().nunique())
print('Setelah diseragamkan hurufnya    :', s.str.strip().str.title().nunique())
print()
print('Sisa yang perlu keputusan manusia:')
print(sorted(s.str.strip().str.title().unique()))

> Dua yang tersisa — `Minumam` dan `Alat Tulis` — **tidak bisa diselesaikan oleh kode**.
> Komputer tidak tahu bahwa `Minumam` adalah salah ketik dari `Minuman`, dan bahwa `Alat Tulis`
> adalah nama lain untuk `ATK`. Keputusan ini harus diambil manusia, lalu **dicatat alasannya**.
> 
> Inilah sebabnya pertemuan 5 mewajibkan *cleaning log*.

In [ ]:
# CONTOH 6.3 — spasi tersembunyi pada kolom kunci
a = df['id_anggota'].astype(str)
print('id_anggota unik apa adanya :', a.nunique())
print('id_anggota unik setelah strip:', a.str.strip().nunique())
print()
print('Selisih', a.nunique() - a.str.strip().nunique(),
      'anggota hantu — muncul hanya karena ada spasi di akhir teks.')

> **Dampaknya nyata.** Kalau Anda menghitung jumlah anggota aktif koperasi tanpa membuang spasi,
> laporan Anda akan menyebut puluhan anggota yang sebenarnya tidak ada. Angkanya rapi, tabelnya cantik,
> dan hasilnya salah.

In [ ]:
# CONTOH 6.4 — periksa seluruh kolom teks sekaligus
for kol in df.select_dtypes(include='object').columns:
    s = df[kol].astype(str)
    mentah = s.nunique()
    rapi   = s.str.strip().str.lower().nunique()
    if mentah != rapi:
        print(f'{kol:20s} {mentah:4d} -> {rapi:4d}   (selisih {mentah - rapi})')

### TUGAS 6 — konsistensi kategori

In [ ]:
# TUGAS 6
t6_kat       = None   # jumlah nilai unik kolom kategori APA ADANYA
t6_katbersih = None   # jumlah nilai unik SETELAH strip, title, dan penyatuan sinonim serta salah ketik

cek('T6_kat', t6_kat)
cek('T6_katbersih', t6_katbersih)

### TUGAS 9 — anggota hantu

In [ ]:
# TUGAS 9
t9_anggota = None   # nunique id_anggota apa adanya
t9_strip   = None   # nunique id_anggota setelah strip

cek('T9_anggota', t9_anggota)
cek('T9_strip', t9_strip)

---
## Bagian 7 — Tipe data yang salah

In [ ]:
# CONTOH 7.1 — uang yang tersimpan sebagai teks
contoh = df['total_bayar'].astype(str)
print('Contoh nilai:', contoh.head(8).tolist())
print()
berformat = contoh.str.startswith('Rp')
print('Nilai berformat "Rp" :', berformat.sum())
print('Nilai berupa angka   :', (~berformat).sum())

> Satu nilai berformat sudah cukup membuat **seluruh kolom** menjadi `object`.
> Selama kolom ini masih teks, `sum()` akan menyambung teks, bukan menjumlahkan uang —
> dan `mean()` akan langsung gagal.

In [ ]:
# CONTOH 7.2 — tanggal dengan dua format bercampur
t = df['tanggal_transaksi'].astype(str)
print('Contoh:', t.head(6).tolist())
print()
iso = t.str.match(r'^\d{4}-')
print('Berformat yyyy-mm-dd :', int(iso.sum()))
print('Berformat dd/mm/yyyy :', int((~iso).sum()))

### Bagian paling berbahaya di seluruh workbook ini

Cara yang paling sering dipakai orang untuk menangani format campur adalah `format='mixed'`
ditambah `dayfirst=True`, karena di Indonesia tanggal memang ditulis hari lebih dulu.
Jalankan sel berikut dan amati baik-baik.

In [ ]:
# CONTOH 7.3 — cara yang tampak benar
cara_umum = pd.to_datetime(t, format='mixed', dayfirst=True, errors='coerce')

print('Gagal dikonversi :', int(cara_umum.isna().sum()))
print('Rentang tanggal  :', cara_umum.min(), 'sampai', cara_umum.max())
print('Bulan yang muncul:', sorted(cara_umum.dt.month.dropna().unique()))

> **Nol galat. Tidak ada peringatan apa pun. Dan hasilnya salah.**
> 
> Dataset ini berisi transaksi **Agustus sampai November 2025** — empat bulan. Hasil di atas
> menyebar ke seluruh dua belas bulan.
> 
> Penyebabnya: `dayfirst=True` ikut diterapkan pada baris yang formatnya sudah `yyyy-mm-dd`.
> Baris `2025-08-01` dibaca sebagai tahun 2025, **hari** 08, **bulan** 01, sehingga menjadi 8 Januari.
> Karena 08 dan 01 keduanya angka yang sah untuk hari maupun bulan, pandas tidak punya alasan mengeluh.
> 
> Inilah jenis kesalahan yang paling mahal dalam analisis data: **salah yang tidak menimbulkan galat.**
> Tidak ada yang berwarna merah, tidak ada yang berhenti, dan laporan Anda tetap terbit.

In [ ]:
# CONTOH 7.4 — cara yang benar: tangani tiap format secara terpisah
benar = pd.Series(pd.NaT, index=t.index, dtype='datetime64[ns]')
benar[iso]  = pd.to_datetime(t[iso],  format='%Y-%m-%d %H:%M:%S', errors='coerce')
benar[~iso] = pd.to_datetime(t[~iso], format='%d/%m/%Y %H:%M',    errors='coerce')

print('Gagal dikonversi :', int(benar.isna().sum()))
print('Rentang tanggal  :', benar.min(), 'sampai', benar.max())
print('Bulan yang muncul:', sorted(benar.dt.month.dropna().unique()))
print()
print('Baris yang berbeda antara cara umum dan cara benar:', int((cara_umum != benar).sum()))

> **Pelajaran yang dibawa pulang:** setelah setiap konversi tanggal, jalankan satu uji akal sehat.
> Yang paling murah adalah memeriksa bulan atau tahun yang muncul:
> 
> ```python
> print(sorted(kolom.dt.month.unique()))
> print(kolom.min(), kolom.max())
> ```
> 
> Kalau rentangnya tidak sesuai dengan yang Anda ketahui tentang data itu, konversinya salah —
> berapa pun jumlah galat yang muncul.

### TUGAS 7 dan 12 — tipe data

In [ ]:
# TUGAS 7
t7_rp = None   # berapa baris yang total_bayar-nya tersimpan dengan awalan "Rp"

cek('T7_rp', t7_rp)

In [ ]:
# TUGAS 12
# Pakai hasil konversi yang BENAR (variabel benar dari CONTOH 7.4).
t12_bulan = None   # berapa bulan berbeda yang muncul di data
t12_rusak = None   # berapa baris yang nilainya berbeda antara cara umum dan cara benar

cek('T12_bulan', t12_bulan)
cek('T12_rusak', t12_rusak)

---
## Bagian 8 — Validitas dan akurasi: aturan yang harus dipenuhi data

Dua pemeriksaan terakhir tidak bisa ditebak dari datanya saja. Keduanya berasal dari **pengetahuan
tentang hal yang diukur**, dan karena itu harus Anda tuliskan sendiri sebagai daftar aturan.

In [ ]:
# CONTOH 8.1 — aturan rentang
aturan = {
    'jumlah harus positif'          : (df['jumlah'] <= 0),
    'diskon harus 0-100'            : (df['diskon_persen'] < 0) | (df['diskon_persen'] > 100),
    'rating harus 1-5'              : (df['rating_layanan'] < 1) | (df['rating_layanan'] > 5),
    'harga satuan harus positif'    : (df['harga_satuan'] <= 0),
}

for nama, pelanggar in aturan.items():
    print(f'{nama:32s} dilanggar {int(pelanggar.sum()):4d} baris')

In [ ]:
# CONTOH 8.2 — aturan hubungan antarkolom
# total_bayar seharusnya = jumlah x harga_satuan x (1 - diskon/100)
# Hanya bisa diperiksa pada baris yang total_bayar-nya sudah berupa angka.
num = df[~df['total_bayar'].astype(str).str.startswith('Rp')].copy()
num['total_angka'] = pd.to_numeric(num['total_bayar'], errors='coerce')
num['seharusnya']  = (num['jumlah'] * num['harga_satuan'] * (1 - num['diskon_persen'] / 100)).round()

tidak_cocok = num['total_angka'] != num['seharusnya']
print('Baris yang bisa diperiksa :', len(num))
print('Tidak cocok               :', int(tidak_cocok.sum()))
print()
num.loc[tidak_cocok, ['jumlah', 'harga_satuan', 'diskon_persen', 'total_angka', 'seharusnya']].head(6)

> **Perhatikan keterbatasannya.** Pemeriksaan ini hanya berjalan pada baris yang total_bayar-nya sudah
> berupa angka. Ratusan baris lain belum bisa diperiksa sampai kolomnya dikonversi pada pertemuan 5.
> 
> Ini pola yang berulang: satu cacat menyembunyikan cacat lain. Karena itu diagnosis dilakukan
> **berlapis**, dan diulang setelah pembersihan.

### TUGAS 8 dan 10 — validitas dan aturan bisnis

In [ ]:
# TUGAS 8
t8_jumlah = None   # berapa baris dengan jumlah <= 0
t8_diskon = None   # berapa baris dengan diskon di luar rentang 0-100

cek('T8_jumlah', t8_jumlah)
cek('T8_diskon', t8_diskon)

In [ ]:
# TUGAS 10
t10_bisnis = None  # berapa baris (dari yang total_bayar-nya numerik) yang melanggar aturan hitungan

cek('T10_bisnis', t10_bisnis)

---
## Bagian 9 — Merangkai temuan menjadi profil kualitas data

Pemeriksaan yang berserakan di banyak sel tidak berguna bagi siapa pun. Rangkum menjadi satu tabel
yang bisa dibaca orang lain dalam satu menit. Tabel inilah inti **M2**.

In [ ]:
# CONTOH 9.1 — kerangka profil kualitas data
temuan = []

def catat(dimensi, kolom, masalah, jumlah, keparahan, rencana):
    temuan.append({
        'dimensi'   : dimensi,
        'kolom'     : kolom,
        'masalah'   : masalah,
        'jumlah'    : int(jumlah),
        'persen'    : round(jumlah / len(df) * 100, 2),
        'keparahan' : keparahan,          # Tinggi / Sedang / Rendah
        'rencana'   : rencana,            # diisi lengkap pada pertemuan 5
    })

catat('Kelengkapan', 'saldo_simpanan', 'Kosong, diduga MNAR',
      df['saldo_simpanan'].isna().sum(), 'Tinggi', 'Jangan diimputasi; tanyakan ke pengurus koperasi')
catat('Kelengkapan', 'berat_kg', 'Kosong untuk kategori non-fisik (MAR)',
      df['berat_kg'].isna().sum(), 'Rendah', 'Tandai tidak berlaku, bukan diisi')
catat('Kelengkapan', 'rating_layanan', 'Placeholder -999',
      (df['rating_layanan'] == -999).sum(), 'Tinggi', 'Ubah menjadi NaN sebelum dihitung')
catat('Keunikan', 'seluruh kolom', 'Baris duplikat persis',
      df.duplicated().sum(), 'Sedang', 'Hapus, simpan satu')
catat('Konsistensi', 'kategori', 'Satu kategori ditulis beberapa cara',
      df['kategori'].astype(str).nunique() - 5, 'Tinggi', 'Seragamkan; sinonim diputuskan manual')
catat('Validitas', 'jumlah', 'Nilai nol atau negatif',
      (df['jumlah'] <= 0).sum(), 'Tinggi', 'Telusuri apakah retur; bila bukan, keluarkan dari analisis')

profil = pd.DataFrame(temuan).sort_values(['keparahan', 'jumlah'], ascending=[True, False])
profil

### TUGAS 11 — lengkapi profilnya

Tambahkan **minimal empat temuan lagi** dengan memanggil `catat(...)`, lalu tampilkan ulang tabelnya.
Kandidat yang belum masuk: placeholder teks pada `metode_bayar`, `total_bayar` bertipe teks,
format tanggal campur, spasi pada `id_anggota`, diskon di luar rentang, pelanggaran aturan hitungan,
dan `id_transaksi` yang kosong atau berulang.

Tidak ada sel `cek` untuk tugas ini — penilaiannya memakai Rubrik M2.

In [ ]:
# TUGAS 11
# catat('...', '...', '...', ..., '...', '...')


profil = pd.DataFrame(temuan).sort_values(['keparahan', 'jumlah'], ascending=[True, False])
profil

In [ ]:
# Simpan profil Anda
profil.to_csv('profil_kualitas_latihan.csv', index=False)
print('Tersimpan:', profil.shape)

---
## Bagian 10 — Terapkan pada data proyek tim

**Ini bagian yang dinilai.** Hasilnya menjadi bagian pertama dari **M2 — Data Quality Report**
yang dikumpulkan pada pertemuan 5.

Kerjakan bersama tim pada dataset proyek Anda sendiri.

In [ ]:
# 10.1 — muat dataset proyek tim
# df_tim = pd.read_csv('data/raw/....csv')
# df_tim.info()


In [ ]:
# 10.2 — jalankan kesembilan pemeriksaan pada data tim
# Salin pola Bagian 2 sampai 8. Untuk setiap pemeriksaan, catat temuannya.
# 1. Potret awal: shape, info, describe
# 2. Nilai kosong sejati
# 3. Placeholder teks dan angka
# 4. Pola kekosongan: MCAR / MAR / MNAR untuk setiap kolom yang banyak kosong
# 5. Duplikat persis dan duplikat menurut kunci
# 6. Konsistensi kolom teks
# 7. Tipe data yang salah
# 8. Aturan rentang
# 9. Aturan hubungan antarkolom


In [ ]:
# 10.3 — susun profil kualitas data tim
# Pakai fungsi catat() yang sama, lalu simpan.
# temuan_tim = []
# ...
# profil_tim.to_csv('reports/profil_kualitas_data.csv', index=False)


### Daftar periksa Bagian 10

- [ ] Seluruh sembilan pemeriksaan dijalankan pada data tim
- [ ] Setiap kolom yang punya nilai kosong di atas 5% sudah dinilai MCAR, MAR, atau MNAR, **beserta alasannya**
- [ ] Minimal lima aturan validitas ditulis sendiri berdasarkan pengetahuan tentang data tim
- [ ] Minimal satu aturan hubungan antarkolom diperiksa
- [ ] Profil kualitas data tersimpan di `reports/`
- [ ] Notebook di-*commit* ke repositori tim dengan pesan yang bermakna

> **Ingat:** hari ini **tidak ada** yang diperbaiki. Perbaikan beserta justifikasinya adalah pekerjaan
> pertemuan 5. Yang diserahkan hari ini adalah diagnosis.

---
## Kalau Anda mentok

Ikuti urutan ini sebelum bertanya:

| Urutan | Yang dilakukan |
|---|---|
| 1 | Baca pesan galat **baris terakhirnya** — di situ nama galat dan penyebabnya |
| 2 | Jalankan ulang dari sel paling atas: *Runtime → Restart and run all* |
| 3 | Periksa kembali petunjuk yang muncul dari sel `cek(...)` |
| 4 | Tanyakan ke anggota tim; kemungkinan besar mereka menemui galat yang sama |
| 5 | Tanyakan ke asisten saat sesi berlangsung |
| 6 | Tulis di **forum tanya-jawab LMS**. Dosen menjawab setiap hari kerja. |

**Cara bertanya yang membuat jawaban datang lebih cepat:** sebutkan nomor bagian, tempelkan baris
terakhir pesan galat, dan sebutkan apa yang sudah Anda coba. Pertanyaan "kodenya error" tanpa keterangan
akan dibalas dengan pertanyaan balik, dan Anda kehilangan satu hari.

---
## Ringkasan pertemuan 4

1. **Lima dimensi:** kelengkapan, keunikan, konsistensi, validitas, akurasi. Periksa berurutan.
2. **Placeholder lolos dari `isna()`.** `-999` dan `N/A` terlihat seperti data sungguhan.
3. **MAR dapat dibuktikan; MCAR dan MNAR tidak.** Membedakannya butuh pengetahuan tentang cara data dikumpulkan.
4. **Duplikat nyaris lebih berbahaya daripada duplikat persis**, karena `duplicated()` tidak menangkapnya.
5. **Spasi tersembunyi menciptakan entitas hantu** dan diam-diam merusak hasil agregasi.
5b. **Konversi tanggal bisa salah tanpa satu pun galat.** Selalu periksa rentang dan bulan hasilnya.
6. **Satu cacat menyembunyikan cacat lain.** Diagnosis diulang setelah pembersihan.
7. **Hari ini mendiagnosis, belum memperbaiki.**

### Yang dikerjakan sebelum pertemuan 5

- Selesaikan Bagian 10 bersama tim dan *commit* ke repositori.
- Kerjakan kuis formatif pertemuan 4 di LMS (boleh diulang sebanyak yang Anda mau).
- Baca Bagian 1 workbook pertemuan 5 agar sesi berikutnya bisa langsung praktik.